# Demo: Using kb-mcp with hep-multiagent

In [ ]:
# !pip install -q --force-reinstall git+https://github.com/HEP-KE/mcp-ke.git
!pip install -q --force-reinstall git+https://github.com/HEP-KE/HEP-multiagent.git
#install last since it needs mcp<1.23.0 but 1.26.0 got installed
!pip install -q --force-reinstall git+https://github.com/HEP-KE/kb-mcp.git

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.0.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.1 which is incompatible.


### Set up for KB MCP

Set up paths and choose which papers to download.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Paths (stores data in current directory)
DATA_DIR = Path.cwd() / "data"
DB_PATH = DATA_DIR / "kb.db"
PAPERS_DIR = DATA_DIR / "papers"

# Papers to download (arXiv ID, title)
PAPERS = [
    ("1807.06209", "Planck 2018 cosmological parameters"),
    ("2007.08991", "eBOSS cosmological results"),
    ("1502.01589", "Planck 2015 cosmological results"),
]

print(f"Database: {DB_PATH}")
print(f"Papers: {PAPERS_DIR}")

Initialize an empty SQLite database with the kb-mcp schema.

In [ ]:
from kb_mcp.kb.db_models import Base
from sqlalchemy import create_engine

# Create directories
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
PAPERS_DIR.mkdir(parents=True, exist_ok=True)

# Create database
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.create_all(engine)

print(f"Created database: {DB_PATH}")

Fetch PDFs from arXiv and extract text.

In [ ]:
from hep_multiagent.features.arxiv_fetch import download_full_text

for arxiv_id, title in PAPERS:
    txt_path = PAPERS_DIR / f"{arxiv_id}.txt"
    if txt_path.exists():
        print(f"{arxiv_id} - already downloaded")
    else:
        print(f"[downloading] {arxiv_id}: {title}")
        download_full_text(arxiv_id, str(PAPERS_DIR))

print(f"\nDownloaded {len(list(PAPERS_DIR.glob('*.txt')))} papers")

Ingest the downloaded papers into kb-mcp.

In [ ]:
# NOTE: Embeddings required for kb_search to work with SQLite.
# Using --no-embed causes kb_search to crash (KeyError: 'total_results')
# because SQLite doesn't support full-text search.

os.environ["SQLITE_DB_PATH"] = str(DB_PATH)

for arxiv_id, _ in PAPERS:
    txt_path = PAPERS_DIR / f"{arxiv_id}.txt"
    if txt_path.exists():
        print(f"[ingesting] {arxiv_id}")
        
        subprocess.run(
            [sys.executable, "-m", "kb_mcp.kb.cli", "ingest", str(txt_path),
             "--source-id", "arxiv", "--no-summary", "--batch"],
            capture_output=True
        )

print("\nDone! Checking database...")
result = subprocess.run([sys.executable, "-m", "kb_mcp.kb.cli", "stats"], capture_output=True, text=True)
print(result.stdout)

### Set up HEP Multiagent

Set up the LLM

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="claudesonnet4",
    base_url="https://apps-dev.inside.anl.gov/argoapi/v1",
    api_key=os.environ["ARGO_USER"]
)
print("Using Argo API")

Initialize hep-multiagent with kb-mcp. The agent gets these tools:
- `kb_search` - search papers by keyword
- `kb_get` - get full text of a paper

In [ ]:
from hep_multiagent import Agent

agent = await Agent(
    llm=llm,
    mcp_servers=[
    {
        "url": "https://github.com/HEP-KE/kb-mcp.git",
        "name": "kb-server-stdio",
        "env": {"SQLITE_DB_PATH": str(DB_PATH)},
    },
    {
        "url": "https://github.com/HEP-KE/mcp-ke.git",
        "env": {
            "LLM_API_KEY": os.environ.get("ARGO_USER", ""),
            "LLM_URL": "https://apps-dev.inside.anl.gov/argoapi/v1",
            "LLM_MODEL": "claudesonnet4",
        },
    },
    ],
    approval=False,
)

for tool in agent.tools:
    print(f"  - {tool.name}")

## Query

In [ ]:
result = await agent.run(
    query = """
Using the observational data from eBOSS DR14 Lyman-alpha forest (./input/DR14_pm3d_19kbins.txt), 
compare the linear P(k) values for ΛCDM, ΛCDM with massive neutrinos (Σmν=0.10 eV), and dark 
energy model with equation of state parameter w0=-0.9. 

Create visualizations showing:
1. The power spectra comparison with observational data
2. The suppression ratios relative to ΛCDM

Comment on how close the P(k) values are and analyze the power spectrum suppression compared to ΛCDM.
""",
    output_dir="./output_kb_hep_mcp"
)

print(result)